# Trace Extraction Visualizer
For each example file, this notebook renders the BPMN diagram and shows how many execution traces are extracted.

In [6]:
# Standard imports
import sys
import os
import json

sys.path.append("../")
sys.path.append("../model_evaluation/")

from IPython.display import display, Markdown

from model_evaluation.BPMN_conversion import BPMNConverter
from model_evaluation.XML_conversion import XMLBPMNConverter
from model_evaluation.petri import PetriNet

from rendering import render_bpmn_xml_embed

In [7]:
def load_model(path):
    """Load a BPMN model from a .json (Signavio) or .xml (BPMN 2.0) file.
    Returns the normalised dict ready for the evaluation pipeline.
    """
    if path.endswith(".xml"):
        return XMLBPMNConverter.convert_file(path).to_dict()
    else:
        with open(path, "r", encoding="utf-8") as fh:
            raw = json.load(fh)
        return BPMNConverter.convert(raw).to_dict()


def get_trace_count(model_json, timeout=5.0, max_loop=3):
    """Extract traces and return the count."""
    try:
        pn = PetriNet.from_simplified_json(model_json)
        traces = pn.net_variants(time_out_sec=timeout, max_loop_depth=max_loop)
        return len(traces), pn
    except Exception as e:
        return None, str(e)


def analyze_file(path, timeout=5.0, max_loop=3):
    """Render the BPMN diagram and print trace info for one file."""
    fname = os.path.basename(path)
    
    # Read raw XML for rendering
    xml_str = None
    if path.endswith(".xml"):
        with open(path, "r", encoding="utf-8") as f:
            xml_str = f.read()
    
    # Load model and extract traces
    try:
        model_json = load_model(path)
        n_activities = len(model_json.get("activities", []))
        n_events = len(model_json.get("events", []))
        n_gateways = len(model_json.get("gateways", []))
        total_elements = n_activities + n_events + n_gateways
        
        result = get_trace_count(model_json, timeout, max_loop)
        if result[0] is not None:
            n_traces, pn = result
            n_places = len(pn.places)
            n_transitions = len(pn.transitions)
            n_arcs = len(pn.arcs)
            status = "\u2705"
        else:
            n_traces = None
            n_places = n_transitions = n_arcs = 0
            status = "\u274c"
            error_msg = result[1]
    except Exception as e:
        status = "\u274c"
        n_traces = None
        total_elements = n_activities = n_events = n_gateways = 0
        n_places = n_transitions = n_arcs = 0
        error_msg = str(e)
    
    # Print file info
    print(f"\n{'=' * 80}")
    print(f"{status}  {fname}")
    print(f"{'=' * 80}")
    
    if n_traces is not None:
        print(f"BPMN elements:  {total_elements} (acts={n_activities}, events={n_events}, gateways={n_gateways})")
        print(f"Petri net:      {n_places} places, {n_transitions} transitions, {n_arcs} arcs")
        print(f"Trace variants: {n_traces}")
    else:
        print(f"BPMN elements:  {total_elements}")
        print(f"ERROR: {error_msg}")
    
    # Render XML diagram
    if xml_str:
        render_bpmn_xml_embed(xml_str, height_px=350, navigated=True)
    else:
        print("(no XML available for rendering)")
    
    return n_traces

---
## Results

In [8]:
# Settings - adjust these as needed
TIMEOUT = 2.0    # seconds per model
MAX_LOOP = 5       # max loop iterations

# Collect all .xml and .json files from the examples directory
example_dir = "../examples"
files = []
for fname in sorted(os.listdir(example_dir)):
    if not (fname.endswith(".xml") or fname.endswith(".json")):
        continue
    if fname.startswith("minimal"):
        continue
    files.append(os.path.join(example_dir, fname))

print(f"Found {len(files)} files to analyze.")

Found 28 files to analyze.


In [9]:
# Run the analysis for each file
results = {}
for fpath in files:
    n = analyze_file(fpath, timeout=TIMEOUT, max_loop=MAX_LOOP)
    fname = os.path.basename(fpath)
    results[fname] = n


✅  01 BPMN Training -T-shirt order simple.xml
BPMN elements:  11 (acts=7, events=2, gateways=2)
Petri net:      12 places, 12 transitions, 24 arcs
Trace variants: 2



✅  02 BPMN Training -T-shirt order (extended ).xml
BPMN elements:  13 (acts=9, events=2, gateways=2)
Petri net:      14 places, 14 transitions, 28 arcs
Trace variants: 2



✅  03 Prepare delivery (subprocess).xml
BPMN elements:  7 (acts=3, events=2, gateways=2)
Petri net:      8 places, 8 transitions, 16 arcs
Trace variants: 2



❌  Adrians_ex.json
BPMN elements:  0
ERROR: Process structure error: Sequence flow sid-373935DE-B64E-4B59-A45D-B6001FF70D10 crosses subprocess boundary (from 'sid-2C01C673-6036-4212-B4BB-212F784403F0' in subprocess 'None' to 'sid-F015C8E1-D461-4980-AAD0-A2F6A7D34AFC' in subprocess 'sid-CEC28D77-4DA4-44A2-B67B-84D343033FD9').
(no XML available for rendering)

✅  E_j04.json
BPMN elements:  9 (acts=6, events=2, gateways=1)
Petri net:      6 places, 13 transitions, 12 arcs
Trace variants: 28186
(no XML available for rendering)

✅  E_j04_4.bpmn2 _ Signavio.json
BPMN elements:  22 (acts=13, events=2, gateways=7)
Petri net:      30 places, 31 transitions, 66 arcs
Trace variants: 29
(no XML available for rendering)

✅  Gateway AND.xml
BPMN elements:  8 (acts=4, events=2, gateways=2)
Petri net:      10 places, 8 transitions, 18 arcs
Trace variants: 2



✅  Gateway Inclusive.xml
BPMN elements:  8 (acts=4, events=2, gateways=2)
Petri net:      8 places, 8 transitions, 16 arcs
Trace variants: 2


...


✅  ls_2.xml
BPMN elements:  6 (acts=4, events=2, gateways=0)
Petri net:      7 places, 6 transitions, 12 arcs
Trace variants: 1



✅  ls_3.xml
BPMN elements:  6 (acts=4, events=2, gateways=0)
Petri net:      7 places, 6 transitions, 12 arcs
Trace variants: 1



✅  ls_4.xml
BPMN elements:  6 (acts=4, events=2, gateways=0)
Petri net:      7 places, 6 transitions, 12 arcs
Trace variants: 1



✅  misc_booking_flight_tickets.json
BPMN elements:  18 (acts=4, events=11, gateways=3)
Petri net:      29 places, 30 transitions, 63 arcs
Trace variants: 4
(no XML available for rendering)

✅  misc_booking_variant.json
BPMN elements:  20 (acts=4, events=12, gateways=4)
Petri net:      33 places, 35 transitions, 73 arcs
Trace variants: 4
(no XML available for rendering)

✅  misc_claim_to_damages.json
BPMN elements:  18 (acts=8, events=7, gateways=3)
Petri net:      25 places, 28 transitions, 56 arcs
Trace variants: 5
(no XML available for rendering)

✅  misc_credit_quote_creation.json
BPMN elements:  11 (acts=6, events=2, gateways=3)
Petri net:      15 places, 14 transitions, 30 arcs
Trace variants: 6
(no XML available for rendering)

✅  misc_loan_brokerage.json
BPMN elements:  20 (acts=5, events=11, gateways=4)
Petri net:      31 places, 34 transitions, 69 arcs
Trace variants: 4
(no XML available for rendering)

✅  misc_purchase_requisition_to_order.json
BPMN elements:  12 (acts=7, ev


✅  test.json
BPMN elements:  12 (acts=3, events=8, gateways=1)
Petri net:      20 places, 21 transitions, 42 arcs
Trace variants: 3
(no XML available for rendering)

✅  test_variant.json
BPMN elements:  26 (acts=7, events=19, gateways=0)
Petri net:      40 places, 64 transitions, 114 arcs
Trace variants: 10931
(no XML available for rendering)


---
## Summary Table

In [10]:
print(f"{'File':<55} {'Traces':<10}")
print("-" * 65)
for fname in sorted(results):
    n = results[fname]
    if n is not None:
        print(f"{fname:<55} {n:<10}")
    else:
        print(f"{fname:<55} {'ERROR':<10}")

success_count = sum(1 for v in results.values() if v is not None)
fail_count = sum(1 for v in results.values() if v is None)
print(f"\n\u2705 Successful: {success_count}   \u274c Failed: {fail_count}")

File                                                    Traces    
-----------------------------------------------------------------
01 BPMN Training -T-shirt order simple.xml              2         
02 BPMN Training -T-shirt order (extended ).xml         2         
03 Prepare delivery (subprocess).xml                    2         
Adrians_ex.json                                         ERROR     
E_j04.json                                              28186     
E_j04_4.bpmn2 _ Signavio.json                           29        
Gateway AND.xml                                         2         
Gateway Inclusive.xml                                   2         
Gateway XOR.xml                                         3         
credit.json                                             6         
credit.xml                                              6         
linear_sequence.json                                    1         
linear_sequence.xml                                     1      